In [1]:
import numpy as np
import pandas as pd
import csv

In [2]:
with open('/content/data.csv', 'r') as file:
    try:
        data = list(csv.reader(file))
    except csv.Error as e:
        print(f'Error reading CSV: {e}')

In [3]:
columns = ['',
  'Movie Name',
  'Year of Release',
  'Run Time in minutes',
  'Movie Rating',
  'Votes',
  'MetaScore',
  'Gross',
  'Genre',
  'Certification',
  'Director',
  'Stars',
  'Description']

In [4]:
data = pd.DataFrame(data, columns=columns)
data.head()

,,Movie Name,Year of Release,Run Time in minutes,Movie Rating,Votes,MetaScore,Gross,Genre,Certification,Director,Stars,Description
0,,Movie Name,Year of Release,Run Time in minutes,Movie Rating,Votes,MetaScore,Gross,Genre,Certification,Director,Stars,Description
1,0,The Shawshank Redemption,1994,142,9.3,2804443,82.0,28340000.0,['Drama'],R,['Frank Darabont'],"['Tim Robbins', 'Morgan Freeman', 'Bob Gunton'...","['Over', 'the', 'course', 'of', 'several', 'ye..."
2,1,The Godfather,1972,175,9.2,1954174,100.0,134970000.0,"['Crime', ' Drama']",R,['Francis Ford Coppola'],"['Marlon Brando', 'Al Pacino', 'James Caan', '...","['Don', 'Vito', 'Corleone,', 'head', 'of', 'a'..."
3,2,Ramayana: The Legend of Prince Rama,1993,135,9.2,12995,,,"['Animation', ' Action', ' Adventure']",PG,"['Ram Mohan', 'Yûgô Sakô', 'Koichi Saski']","['Arun Govil', 'Nikhil Kapoor', 'Edie Mirman',...","['An', 'anime', 'adaptation', 'of', 'the', 'Hi..."
4,3,The Chaos Class,1975,87,9.2,42231,,,"['Comedy', ' Drama']",,['Ertem Egilmez'],"['Kemal Sunal', 'Münir Özkul', 'Halit Akçatepe...","['Lazy,', 'uneducated', 'students', 'share', '..."


In [5]:
data = data.drop('', axis=1)
data = data.drop(data.index[0])

In [6]:
data['user_id'] = np.random.randint(1, 10001, size=len(data))
data['movie_id'] = np.random.randint(1, 10001, size=len(data))

In [7]:
data['user_id'] = data['user_id'].astype(int)
data['movie_id'] = data['movie_id'].astype(int)

In [8]:
ratings = data.loc[:, ['movie_id', 'user_id', 'Movie Rating', 'Movie Name']]
ratings.rename(columns={'Movie Rating': 'rating', 'Movie Name': 'movie_title'}, inplace=True)
ratings['rating'] = ratings['rating'].astype(float)
ratings = np.array(ratings)

In [9]:
K = 10 # Número de fatores latentes
alpha = 0.01 # Taxa de aprendizado
beta = 0.01 # Termo de regularização
epochs = 100 # Número de épocas

In [13]:
def rmse():
  # Inicializar o erro como zero
  error = 0
  # Para cada avaliação no conjunto de dados
  for i in range(len(ratings)):
    # Obter o usuário, o filme e a avaliação
    user = int(ratings[i, 1]) - 1
    item = int(ratings[i, 0]) - 1
    rating = ratings[i, 2]
    # Calcular a avaliação estimada como o produto escalar dos fatores latentes
    pred_rating = np.dot(P[user, :], Q[item, :].T)
    # Adicionar o erro quadrático ao erro total
    error += (rating - pred_rating) ** 2
  # Retornar a raiz quadrada do erro médio
  return np.sqrt(error / len(ratings))

In [ ]:
num_users = int(len(ratings))
num_items = int(len(ratings))


# Inicializar as matrizes de fatores latentes de forma aleatória
P = np.random.normal(scale=1./K, size=(num_users, K))
Q = np.random.normal(scale=1./K, size=(num_items, K))

for epoch in range(epochs):
    # Para cada avaliação no conjunto de dados
    for i in range(len(ratings)):
        user_idx, item_idx, rating = int(ratings[i, 1]) - 1, int(ratings[i, 0]) - 1, ratings[i, 2]

        # Calcular a avaliação estimada como o produto escalar dos fatores latentes
        pred_rating = np.dot(P[user_idx, :], Q[item_idx, :].T)

        # Calcular o erro entre a avaliação real e a estimada
        error = rating - pred_rating

        # Atualizar os fatores latentes do usuário e do filme
        P[user_idx, :] += alpha * (error * Q[item_idx, :] - beta * P[user_idx, :])
        Q[item_idx, :] += alpha * (error * P[user_idx, :] - beta * Q[item_idx, :])

    print(f"Epoch {epoch + 1}: RMSE = {rmse()}")

In [15]:
def recommend_items(user_id, n_items):
    # Obter as avaliações do usuário
    user_ratings = ratings[ratings[:, 2] == user_id]

    # Filtrar os filmes que o usuário já avaliou
    rated_items = set(user_ratings[:, 0])

    # Calcular as avaliações estimadas para os filmes não avaliados pelo usuário
    estimated_ratings = [(item_id, np.dot(P[user_id - 1, :], Q[item_id - 1, :].T))
                         for item_id in range(1, num_items + 1) if item_id not in rated_items]

    # Ordenar a lista em ordem decrescente de avaliação
    estimated_ratings.sort(key=lambda x: x[1], reverse=True)

    # Mapear os IDs dos filmes para seus nomes
    movie_names = {int(movie_id): movie_name for movie_id, movie_name in zip(ratings[:, 0], ratings[:, 3])}

    # Retornar os n filmes com maior avaliação estimada, incluindo o nome do filme
    recommended_movies = [(movie_names[item_id], pred_rating) for item_id, pred_rating in estimated_ratings[:n_items]]

    return recommended_movies

In [16]:
# Testar a função para um usuário e um número de itens
recommend_items(user_id=1, n_items=5)

[('The Memory of a Killer', 7.976203912182877),
 ('Splash', 7.200751216795686),
 ('F9: The Fast Saga', 6.847186890834312),
 ('Cobweb', 6.571784802445543),
 ('Wild', 6.395487248036392)]